### EDA

In [18]:
import pandas as pd
import numpy as np

# list of  weather underground files (add more  needed)
files = [
    "Weather underground data/Weather Underground Boston Full 2023.csv",
    "Weather underground data/Weather Underground Boston Full 2024.csv",
    "Weather underground data/Weather Underground Boston Jan-Nov 2025.csv",
]

df_list = []

for file in files:
    df_raw = pd.read_csv(file, low_memory=False)
    print(f"Loaded {file} with shape {df_raw.shape}")
    
    # normalize fog column names just in case (even if we dont use them now)
    rename_dict = {}
    for col in df_raw.columns:
        col_norm = col.replace(" ", "").lower()
        if col_norm == "dew/tempfog?":
            rename_dict[col] = "Dew/Temp fog?"
        if col_norm == "rhfog?":
            rename_dict[col] = "RH fog?"
    if rename_dict:
        df_raw = df_raw.rename(columns=rename_dict)
    
    df_list.append(df_raw)

# single raw dataframe with all years
df_raw = pd.concat(df_list, ignore_index=True)

display(df_raw.head(3))
df_raw.shape


Loaded Weather underground data/Weather Underground Boston Full 2023.csv with shape (111813, 15)
Loaded Weather underground data/Weather Underground Boston Full 2024.csv with shape (143192, 15)
Loaded Weather underground data/Weather Underground Boston Jan-Nov 2025.csv with shape (88331, 15)


,Date,Time,Temperature_C,Dew_Point_C,Humidity_%,Wind,Speed_kmh,Gust_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,UV,Solar_w/m2,Dew/Temp fog?,RH fog?
0,12/1/2023,12:04 AM,4.72,1.17,78,SSW,1.93,2.74,1003.73,0,0,0,0,0,0
1,12/1/2023,12:09 AM,4.89,1.22,77,SSE,1.29,2.25,1003.39,0,0,0,0,0,0
2,12/1/2023,12:14 AM,4.89,1.28,78,South,1.29,2.41,1003.39,0,0,0,0,0,0


(343336, 15)

In [19]:
# combine Date + Time into local Boston time
dt_str = df_raw["Date"].astype(str) + " " + df_raw["Time"].astype(str)
ts_local = pd.to_datetime(dt_str, errors="coerce")

# localize to America/New_York and convert to UTC
ts_utc = (
    ts_local
    .dt.tz_localize("America/New_York", ambiguous="NaT", nonexistent="NaT")
    .dt.tz_convert("UTC")
)

df_raw["timestamp_utc"] = ts_utc
df_raw = df_raw.dropna(subset=["timestamp_utc"]).copy()

# Choose only column required
df = df_raw[[
    "timestamp_utc",
    "Temperature_C",
    "Dew_Point_C",
    "Wind",
    "Speed_kmh",
    "Pressure_hPa",
    "Precip_Rate_mm",
    "Precip_Accum_mm",
]].copy()

df.info()


C:\Users\USER\AppData\Local\Temp\ipykernel_14672\646770081.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts_local = pd.to_datetime(dt_str, errors="coerce")


<class 'pandas.core.frame.DataFrame'>
Index: 343262 entries, 0 to 343335
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   timestamp_utc    343262 non-null  datetime64[ns, UTC]
 1   Temperature_C    343262 non-null  object             
 2   Dew_Point_C      343262 non-null  object             
 3   Wind             336984 non-null  object             
 4   Speed_kmh        343262 non-null  object             
 5   Pressure_hPa     343262 non-null  object             
 6   Precip_Rate_mm   343262 non-null  object             
 7   Precip_Accum_mm  343262 non-null  object             
dtypes: datetime64[ns, UTC](1), object(7)
memory usage: 23.6+ MB


In [ ]:
# ensure numeric typing for some required variables
columns = [
    "Temperature_C",
    "Dew_Point_C",
    "Speed_kmh",
    "Pressure_hPa",
    "Precip_Rate_mm",
    "Precip_Accum_mm",
]

for col in columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 343262 entries, 0 to 343335
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   timestamp_utc    343262 non-null  datetime64[ns, UTC]
 1   Temperature_C    343262 non-null  float64            
 2   Dew_Point_C      343262 non-null  float64            
 3   Wind             336984 non-null  object             
 4   Speed_kmh        343262 non-null  float64            
 5   Pressure_hPa     343262 non-null  float64            
 6   Precip_Rate_mm   343262 non-null  float64            
 7   Precip_Accum_mm  343262 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(6), object(1)
memory usage: 23.6+ MB


### Resampling
This dataset is not at hourly resolution, so we need to resample it to hourly resolution.

In [21]:
# set index for resampling
df = df.set_index("timestamp_utc").sort_index()

# aggregate to hourly resolution
hourly = (
    df
    .resample("1H")
    .agg({
        "Temperature_C": "mean",
        "Dew_Point_C": "mean",
        "Speed_kmh": "mean",
        "Pressure_hPa": "mean",
        "Precip_Rate_mm": "mean", 
        "Precip_Accum_mm": "max", # this is cumulative; hourly max is a dumb down
        "Wind": "last", # wind direction is categorical; we take the last observation in the hour for now, change later if needed
    })
    .reset_index()
)
print(hourly.shape)
hourly.head()

(25152, 8)


C:\Users\USER\AppData\Local\Temp\ipykernel_14672\1182267434.py:7: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample("1H")


,timestamp_utc,Temperature_C,Dew_Point_C,Speed_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,Wind
0,2023-01-01 05:00:00+00:00,12.206667,11.985833,0.440833,989.107500,1.5250,1.02,SSE
1,2023-01-01 06:00:00+00:00,12.106667,11.884167,0.506667,988.518333,0.3175,1.27,SE
2,2023-01-01 07:00:00+00:00,11.795833,11.595833,0.093333,987.640000,0.0000,1.27,SSE
3,2023-01-01 08:00:00+00:00,11.620000,11.404167,0.280000,988.008333,0.4450,1.52,SSE
4,2023-01-01 09:00:00+00:00,11.005000,10.755000,0.240000,988.178333,0.0000,1.52,SSE


In [22]:
hourly.to_csv("Weather underground data/Weather_Underground_hourly_full.csv", index=False)
